In [1]:
import pandas as pd
import ast
print(pd.__version__)

3.0.3


In [41]:
# Cargar el archivo original
df = pd.read_pickle('metadatos_guion_adaptado.pkl')
df.rename(columns={'Vote_Average': 'Rating_Score', 'Vote_Count' : 'Votes_Count'}, inplace=True)
# df = df.drop(columns=['Male_Characters_Count', 'Female_Characters_Count', 'Unknown_Characters_Count'])
df.to_pickle('metadatos_guion_adaptado.pkl')
df.to_csv('metadatos_guion_adaptado.csv', index=False)

In [42]:
df = pd.read_pickle('metadatos_guion_adaptado.pkl')


# Mover columnas de Director
col = df.pop("gender_director")
df.insert(df.columns.get_loc("Director") + 1, "gender_director", col)

col = df.pop("male_director")
df.insert(df.columns.get_loc("gender_director") + 1, "male_director", col)

col = df.pop("female_director")
df.insert(df.columns.get_loc("male_director") + 1, "female_director", col)

# Mover columnas de Writers
col = df.pop("male_writer")
df.insert(df.columns.get_loc("Writers") + 1, "male_writer", col)

col = df.pop("female_writer")
df.insert(df.columns.get_loc("male_writer") + 1, "female_writer", col)

# Guardar
df.to_pickle("metadatos_guion_adaptado_2.pkl")
df.to_csv("metadatos_guion_adaptado_2.csv", index=False, encoding="utf-8-sig")

In [44]:
# Cargar los 3 datasets
df1 = pd.read_pickle('metadatos_mejor_pelicula.pkl')
df2 = pd.read_pickle('metadatos_guion_original.pkl')
df3 = pd.read_pickle('metadatos_guion_adaptado.pkl')

# print(set(df1.columns) - set(df2.columns))
# print(set(df2.columns) - set(df1.columns))
# print(set(df1.columns) - set(df3.columns))
# print(set(df1.columns) - set(df2.columns))
# print(set(df2.columns) - set(df3.columns))
# print(set(df1.columns) - set(df3.columns))

# Concatenar uno encima del otro
df_final = pd.concat([df1, df2, df3], ignore_index=True)

# Guardar resultados
df_final.to_pickle('metadatos_final.pkl')
df_final.to_csv('metadatos_final.csv', index=False)

In [32]:
# 1. Agregar la columna Premio
df['Premio'] = "Mejor Película"

def calcular_estadisticas_genero(row):
    script = row['Script_Dict']
    genders = row['Characters_Genders']
    
    # Asegurar que Script_Dict sea un diccionario (si viene cpkl string)
    if isinstance(script, str):
        try:
            script_dict = ast.literal_eval(script)
        except (ValueError, SyntaxError):
            script_dict = {}
    elif isinstance(script, dict):
        script_dict = script
    else:
        script_dict = {}
    
    # Asegurar que Character_Genders sea un diccionario
    if not isinstance(genders, dict):
        genders = {}
        
    p_hombre = 0
    p_mujer = 0
    p_unknown = 0
    c_hombre = 0
    c_mujer = 0
    c_unknown = 0
    
    # Iterar sobre los personajes que aparecen en el Script_Dict
    for char, dialogo in script_dict.items():
        if dialogo is None:
            continue
            
        # Conteo de palabras del diálogo (usando split para separar por espacios)
        word_count = len(str(dialogo).split())
        
        # Obtener el género del personaje (por defecto 'unknown' si no existe)
        gender = genders.get(char, 'unknown').lower()
        
        if gender == 'male':
            p_hombre += word_count
            c_hombre += 1
        elif gender == 'female':
            p_mujer += word_count
            c_mujer += 1
        elif gender == 'unknown':
            p_unknown += word_count
            c_unknown += 1
            
    # Calcular medias (evitando división por cero)
    media_hombre = round(p_hombre / c_hombre if c_hombre > 0 else 0,2)
    media_mujer = round(p_mujer / c_mujer if c_mujer > 0 else 0,2)
    media_unknown = round(p_unknown / c_unknown if c_unknown > 0 else 0,2)
    
    return c_hombre, c_mujer, c_unknown, p_hombre, p_mujer, p_unknown, media_hombre, media_mujer, media_unknown

In [33]:
# Aplicar la función a cada fila y crear las nuevas columnas
resultados = df.apply(calcular_estadisticas_genero, axis=1)
df[['Male_actor', 'Female_actresses', 'Unknown_actors', 'Words_Male', 'Words_Female', 'Words_Unknown', 'AverageWords_male', 'AverageWords_female', 'AverageWords_unknown']] = pd.DataFrame(resultados.tolist(), index=df.index)

In [34]:
# Guardar los resultados en los archivos indicados
df.to_pickle('metadatos_mejor_pelicula_3.pkl')
df.to_csv('metadatos_mejor_pelicula_3.csv', index=False)

print("Proceso completado. Archivos guardados como dataset_genero_final_2.pkl y .csv")

Proceso completado. Archivos guardados como dataset_genero_final_2.pkl y .csv
